# 11장 실습 ① — 무엇으로 표현하고, 어떻게 읽는가

**TensorFlow 판**

텍스트를 네 가지 방식으로 표현하고 성능을 견줍니다.

| 모델 | 무엇으로 표현하는가 | 순서를 보는가 |
|---|---|:--:|
| 단어 가방 + Dense | 등장 횟수 | ✗ |
| 임베딩 + 평균 | 학습된 벡터 | ✗ |
| 임베딩 + Conv1D | 학습된 벡터 | 이웃 3개 |
| 임베딩 + LSTM | 학습된 벡터 | 처음부터 끝까지 |

**앞의 둘과 뒤의 둘 사이에서 성능이 갈립니다.** 그 이유가 이 장입니다.

## 11.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 11.1 실험대 — 어순이 정답을 바꾸는 감성 분류

IMDB나 네이버 영화평은 **단어 가방만으로도 0.85 이상**이 나옵니다.
그러면 *"순서를 봐야 한다"* 는 이 장의 논증이 숫자로 드러나지 않습니다.

그래서 **어순이 정답을 바꾸도록** 만든 데이터를 씁니다.
인터넷 없이 만들어집니다. (3장 §3.2)

In [ ]:
# 어순이 정답을 바꾸는 감성 분류. 인터넷 없이 만듭니다. (본문 §11.4)
V = len(data.TOY_VOCAB)
x, y = data.toy_reviews(n=8000, length=16, seed=42)
sp = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(sp.summary())
print(f"어휘 {V}개:", " ".join(data.TOY_VOCAB))
print()

for k in range(4):
    print(f"  {data.toy_decode(x[k]):<48} → {'긍정' if y[k] else '부정'}")

print()
print("★ 규칙 — 문장에 감성 단어 하나와 미끼 단어 하나가 있고,")
print("  부정어 '안'이 **둘 중 하나 앞에** 붙습니다.")
print("  '안'이 감성 단어에 붙었으면 정답이 뒤집힙니다.")
print()
print("  영화 안 좋다 정말 배우   → 부정")
print("  영화 좋다 안 정말 배우   → 긍정")
print()
print("  **두 문장에 든 단어의 집합은 같습니다.** 다른 것은 순서뿐입니다.")
print("  그래서 단어 가방으로는 **원리적으로** 풀 수 없습니다.")


def to_bag(seqs, vocab_size=V):
    """단어 가방 — 각 단어가 몇 번 나왔는지만 센다. **순서를 버린다.**"""
    b = np.stack([np.bincount(r, minlength=vocab_size) for r in seqs])
    b[:, 0] = 0                      # <pad>는 세지 않는다
    return b.astype("float32")

## 11.2 모델 정의 — 여기만 판마다 다릅니다

**PyTorch 판에서 `Conv1d` 앞에 `transpose(1, 2)` 가 있는 것**에
주목하십시오. PyTorch는 (배치, **채널**, 길이) 를 받고 Keras는
(배치, 길이, **채널**) 을 받습니다. 자주 틀리는 자리입니다.

In [ ]:
import tensorflow as tf

L_ = tf.keras.layers

def _build(kind, V, L):
    """모델 정의 — **이 함수만 판마다 다릅니다.**"""
    if kind == "bow":
        return tf.keras.Sequential([
            L_.Input(shape=(V,)),
            L_.Dense(16, activation="relu"),
            L_.Dense(1, activation="sigmoid")])
    ls = [L_.Input(shape=(L,)), L_.Embedding(V, 8)]
    if kind == "avg":
        ls += [L_.GlobalAveragePooling1D(), L_.Dense(16, activation="relu")]
    elif kind == "cnn":
        ls += [L_.Conv1D(16, 3, activation="relu"), L_.GlobalMaxPooling1D()]
    elif kind == "lstm":
        ls += [L_.LSTM(32)]
    ls += [L_.Dense(1, activation="sigmoid")]
    return tf.keras.Sequential(ls)

_FIT = {}

def train_text(kind, sp, lr=0.003, seed=42, epochs=25):
    """(시험 정확도, 파라미터 수, 임베딩 표) 를 돌려준다."""
    dlbook.set_seed(seed)
    V, L = len(data.TOY_VOCAB), sp.x_train.shape[1]
    prep = (lambda z: to_bag(z, V)) if kind == "bow" else (lambda z: z)
    m = _build(kind, V, L)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr, clipnorm=1.0),
              loss="binary_crossentropy")
    m.fit(prep(sp.x_train), sp.y_train,
          validation_data=(prep(sp.x_val), sp.y_val),
          epochs=dlbook.smoke.epochs(epochs), batch_size=64, verbose=0)
    _FIT[kind] = (m, prep)
    pred = (m.predict(prep(sp.x_test), verbose=0).reshape(-1) > 0.5).astype(int)
    emb = None
    for lyr in m.layers:
        if isinstance(lyr, L_.Embedding):
            emb = lyr.get_weights()[0]
    return metrics.accuracy(sp.y_test, pred), m.count_params(), emb

def predict_acc(kind, sp, xs):
    """이미 학습된 모델로 임의의 입력에 대한 정확도를 잰다."""
    m, prep = _FIT[kind]
    pred = (m.predict(prep(xs), verbose=0).reshape(-1) > 0.5).astype(int)
    return metrics.accuracy(sp.y_test, pred)

## 11.3 네 모델을 나란히

In [ ]:
kinds = [("bow",  "단어 가방 + Dense"),
         ("avg",  "임베딩 + 평균"),
         ("cnn",  "임베딩 + Conv1D"),
         ("lstm", "임베딩 + LSTM")]

base_acc = {}
print(f"{'모델':<22}{'파라미터':>12}{'시험 정확도':>14}")
print("-" * 48)
for kind, ko in kinds:
    acc, n_params, _ = train_text(kind, sp)
    base_acc[kind] = acc
    print(f"{ko:<22}{n_params:>12,}{acc:>14.3f}")
    dlbook.record(f"ch11_{kind}_acc", acc)
    dlbook.record(f"ch11_{kind}_params", n_params)

print()
print("→ 앞의 둘은 **동전 던지기**입니다. 어순을 버렸으니 풀릴 수 없습니다.")
print("→ 순서를 보는 층을 올리는 순간 풀립니다.")
print("   데이터도, 임베딩도, 어휘도 그대로입니다. **읽는 방식만 바꿨습니다.**")
print("→ **파라미터가 성능을 만들지 않습니다.** Conv1D는 단어 가방보다 크지만")
print("   그 차이는 172개뿐이고, 정확도는 0.51에서 1.00으로 갑니다.")

## 11.4 확인 — 평균은 정말 순서를 지우는가

§11.7의 주장을 직접 확인합니다. **시험 문장의 단어를 무작위로 섞어서**
넣어 봅니다. 순서를 안 보는 모델이라면 **성능이 그대로여야** 합니다.

In [ ]:
# 확인 — 평균이 정말 순서를 지우는가. 시험 문장의 단어를 섞어서 넣습니다.
rng = np.random.default_rng(0)
x_shuf = np.stack([rng.permutation(r) for r in sp.x_test])

print(f"{'모델':<8}{'원래 순서':>12}{'섞은 뒤':>12}{'떨어진 폭':>12}")
print("-" * 44)
for kind in ("avg", "cnn", "lstm"):
    train_text(kind, sp)
    a_ord = predict_acc(kind, sp, sp.x_test)
    a_shuf = predict_acc(kind, sp, x_shuf)
    print(f"{kind:<8}{a_ord:>12.3f}{a_shuf:>12.3f}{a_ord - a_shuf:>12.3f}")
    dlbook.record(f"ch11_shuffle_drop_{kind}", float(a_ord - a_shuf))

print()
print("→ 평균 모델은 **섞어도 그대로**입니다. 애초에 순서를 안 봤기 때문입니다.")
print("→ Conv1D와 LSTM은 무너집니다. **순서를 보고 있었다는 증거**입니다.")

## 11.5 심화 — 규칙을 어렵게 하면 Conv1D가 밀립니다

여기까지는 Conv1D와 LSTM이 똑같이 1.000이었습니다.
**과제가 「붙어 있는 두 단어」로 풀렸기 때문입니다.**

규칙을 바꿔 **먼 거리의 순서 관계**를 묻게 하면 둘이 갈립니다.

In [ ]:
# 심화 — 규칙을 더 어렵게 바꾸면 Conv1D와 LSTM이 갈립니다.
# hard=True: 감성 단어가 **둘**(부호 반대) 들어가고, 정답은 **먼저 나온 쪽**을 따릅니다.
xh, yh = data.toy_reviews(n=8000, length=16, seed=42, hard=True)
sh = data.split(xh, yh, val_ratio=0.2, test_ratio=0.2, seed=42)

for k in range(3):
    print(f"  {data.toy_decode(xh[k]):<52} → {'긍정' if yh[k] else '부정'}")
print()

print(f"{'모델':<8}{'기본 규칙':>12}{'어려운 규칙':>14}")
print("-" * 36)
for kind, base in [("cnn", "cnn"), ("lstm", "lstm")]:
    acc_h, _, _ = train_text(kind, sh)
    print(f"{kind:<8}{base_acc[base]:>12.3f}{acc_h:>14.3f}")
    dlbook.record(f"ch11_hard_{kind}_acc", acc_h)

print()
print("→ **Conv1D가 처음으로 밀립니다.** 창 크기 3으로는 '안 좋다' 같은")
print("   **붙어 있는** 조합만 봅니다. 두 감성 단어 사이에 채움말이 여럿")
print("   끼어 있으면 어느 쪽이 먼저인지 한 창에 담기지 않습니다.")
print("→ LSTM은 처음부터 끝까지 읽으므로 거리와 무관합니다.")

## 정리

| 모델 | 순서를 보는가 | 결과 |
|---|:--:|---|
| 단어 가방 + Dense | ✗ | 동전 던지기 |
| 임베딩 + 평균 | ✗ | 동전 던지기 |
| 임베딩 + Conv1D | 이웃 3개 | 풀림 |
| 임베딩 + LSTM | 전체 | 완전히 풀림 |

- **임베딩은 「무엇으로 표현할지」만 해결합니다.** 순서를 읽으려면
  그 위에 층이 더 필요합니다.
- **평균은 덧셈이고, 덧셈은 순서를 따지지 않습니다.**
  임베딩을 통과한 뒤에 다시 단어 가방이 된 것입니다.
- **파라미터 수가 성능을 만들지 않습니다.** 가장 못한 모델이 가장 큽니다.

### 연습

1. `Conv1D` 의 `kernel_size` 를 3 → 5 → 7로 늘리십시오.
   LSTM에 얼마나 가까워집니까. **왜 그렇습니까.**
2. `toy_reviews(length=32)` 로 문장을 길게 만들고 다시 돌리십시오.
   어느 모델이 가장 많이 떨어집니까.
3. `GlobalAveragePooling1D` 를 `Flatten` 으로 바꾸면 어떻게 됩니까.
   파라미터 수는 어떻게 됩니까.
4. **[열린 문제]** 네이버 영화평(NSMC)이나 IMDB로 같은 표를 만드십시오.
   **단어 가방이 0.5보다 훨씬 높게 나올 것입니다.** 왜입니까.